# Notebook 05 — AI-written legitimate false alarms + LLM rewriting robustness (E01)

**CPU-first / Kaggle Run All / research benchmark, not a claim of verified human annotations.**

Question: Does a phishing classifier trained on source-documented original messages mistakenly flag legitimate emails rewritten by GPT-4o/DeepSeek more often than legitimate originals from the **same held-out parent IDs**? All LLM rewrite rows are **derived**; they are not independent external datasets. Original `Valid` and `Phishing` are **source-assigned** classes, not independently adjudicated ground truth.

Safety and science: `Spam`, `Phishing Simulation` and conflicting/unknown classes excluded. The D2 original/rewrites are joined on `No.`, and **no parent may cross train/validation/test**. Exact normalized text duplication between train and held-out groups is quarantined. No test-derived threshold fitting, public raw text, or per-record outputs. D2/D3 documented common lineage prohibits use of D3 as an independent test. Optional D4 is deliberately excluded from this first matched-parent experiment owing to source/generator and language confounds.

**Output:** aggregate JSON, CSV metrics, human-readable summary, run manifest under `/kaggle/working/phishing_n05_safe/`. Raw source and per-row data remain in `/kaggle/temp/phishing_n05_private/` or verified private Kaggle Input. **No GPU required.** Internet can be disabled if exact pinned source files already exist in a private Kaggle Input.


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, precision_score,
    recall_score, average_precision_score, roc_auc_score, brier_score_loss, confusion_matrix)
from sklearn.calibration import CalibratedClassifierCV
import csv, hashlib, json, os, platform, re, shutil, sys, tempfile, time, urllib.parse, urllib.request
import numpy as np

RELEASE='N05-E01-v1.0-2026-11-02'
D2_REV='d13e9bb73d03f458e6d5b6cb6c7b0a708ea47e3d'
SEED=2026
ROOT=Path('/kaggle/working') if Path('/kaggle').exists() else Path.cwd()
PRIV=(Path('/kaggle/temp') if Path('/kaggle').exists() else Path(tempfile.gettempdir()))/'phishing_n05_private'
OUT=ROOT/'phishing_n05_safe'; OUT.mkdir(parents=True,exist_ok=True); PRIV.mkdir(parents=True,exist_ok=True)
SOURCE={
 'original':('1_DataSet/cleaned_emails.json',7370034,'b9d6dd2ac3d6e2d337b9b9d79fc8f1f4377ea4f9'),
 'deepseek':('3_RePhrase/DeepSeek/emails_rephrased_clean_deepseek.json',7528880,'11cc02aeb03b5a318f3107a71a20f2406b8f790a'),
 'gpt4o':('3_RePhrase/GPT4/emails_rephrased_clean_openai4o.json',7079916,'d0cf5fe3bdf0a3dd962492720c3cb10381f9e313'),
 'randomapi':('3_RePhrase/RandomAPI/emails_rephrased_openrouter_clean.json',7124691,'2febaf0e74ac23689889164bfe5b8e538b706140')
}

def git_blob(p):
    h=hashlib.sha1(b'blob '+str(p.stat().st_size).encode()+b'\x00')
    with p.open('rb') as f:
        for b in iter(lambda:f.read(1<<20),b''):h.update(b)
    return h.hexdigest()

def sha256(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for b in iter(lambda:f.read(1<<20),b''):h.update(b)
    return h.hexdigest()

def acquire(name):
    rel,size,blob=SOURCE[name]; filename=Path(rel).name
    candidates=[]
    if Path('/kaggle/input').exists():candidates=list(Path('/kaggle/input').rglob(filename))
    candidates.append(PRIV/filename)
    for path in candidates:
        if path.is_file() and path.stat().st_size==size and git_blob(path)==blob:
            return path,{'sha256':sha256(path),'bytes':size,'origin':'verified_cache'}
    dst=PRIV/filename; partial=PRIV/(filename+'.part')
    url='https://raw.githubusercontent.com/HarmJ0y/PhishingSpamDataSet/'+D2_REV+'/'+urllib.parse.quote(rel,safe='/')
    try:
        with urllib.request.urlopen(url,timeout=90) as response, partial.open('wb') as f:
            count=0
            while True:
                b=response.read(1<<20)
                if not b:break
                count+=len(b)
                if count>size:raise ValueError('download exceeds pinned size')
                f.write(b)
        if partial.stat().st_size!=size or git_blob(partial)!=blob:raise ValueError('Pinned Git blob mismatch: '+name)
        partial.replace(dst)
    finally:
        if partial.exists():partial.unlink()
    return dst,{'sha256':sha256(dst),'bytes':size,'origin':'pinned_download'}

def get(row,*keys):
    normalized={re.sub('[^a-z0-9]','',str(k).lower()):v for k,v in row.items()}
    for k in keys:
        x=normalized.get(re.sub('[^a-z0-9]','',k.lower()))
        if x is not None:return x
    return None

def load(path):
    with path.open(encoding='utf-8-sig') as f:d=json.load(f)
    if isinstance(d,dict):
        d=next((d[k] for k in ('emails','data','records','items','dataset','results') if isinstance(d.get(k),list)),None)
    if not isinstance(d,list) or not all(isinstance(x,dict) for x in d):raise ValueError('Invalid JSON list')
    return d

def ident(x):
    if x is None or isinstance(x,(dict,list,bool)):return None
    s=str(x).strip()
    if not s or s.casefold() in ('none','null','nan'):return None
    # Source IDs may have numeric JSON and numeric-looking strings.
    if re.fullmatch(r'\d+\.0',s):s=s[:-2]
    return s

def label(r):
    x=get(r,'Type')
    return {'Phishing':1,'Valid':0}.get(x)

def txt(r):return str(get(r,'Subject') or '')+'\n'+str(get(r,'Body') or '')
def norm(s):return ' '.join(s.casefold().split())
def digest_text(s):return hashlib.sha256(norm(s).encode('utf-8')).hexdigest()
def group_index(rows):
    out={}
    for r in rows:
        key=ident(get(r,'No.'))
        if key is None or key in out:raise ValueError('Missing/duplicate original parent ID')
        out[key]=r
    return out

def threshold_at_fpr(y,s,target=0.05):
    # Higher score => phishing. Strict > score_max rejects all; deterministic selection only on val.
    neg=np.asarray(s)[np.asarray(y)==0]
    if len(neg)<20:raise ValueError('Insufficient validation legitimate controls')
    candidates=np.unique(np.r_[np.nextafter(max(s),np.inf),np.asarray(s)])
    best=max(s)+1.0
    for t in candidates:
        fpr=float(np.mean(neg>=t))
        if fpr<=target:best=float(t);break
    return best

def score_metrics(y,s,t):
    y=np.asarray(y,dtype=int);s=np.asarray(s,dtype=float)
    if len(y)==0 or not np.all(np.isfinite(s)):raise ValueError('Invalid/nonfinite predictions')
    pred=(s>=t).astype(int);cm=confusion_matrix(y,pred,labels=[0,1]);tn,fp,fn,tp=cm.ravel()
    bins=np.linspace(0,1,11);ece=0.0
    for b in range(10):
        mask=(s>=bins[b]) & ((s<bins[b+1]) if b<9 else (s<=bins[b+1]))
        if mask.any():ece+=mask.mean()*abs(float(y[mask].mean())-float(s[mask].mean()))
    return {'n':int(len(y)),'positive':int(y.sum()),'legitimate':int(len(y)-y.sum()),
        'precision':float(precision_score(y,pred,zero_division=0)),
        'recall':float(recall_score(y,pred,zero_division=0)),
        'f1':float(f1_score(y,pred,zero_division=0)),
        'fpr':float(fp/(fp+tn)) if fp+tn else None,
        'tp':int(tp),'tn':int(tn),'fp':int(fp),'fn':int(fn),
        'roc_auc':float(roc_auc_score(y,s)) if len(set(y))==2 else None,
        'average_precision':float(average_precision_score(y,s)) if len(set(y))==2 else None,
        'brier':float(brier_score_loss(y,s)), 'ece_10_bins':float(ece)}

def smoke():
    assert ident(3.0)=='3'
    assert norm(' A  B ')=='a b'
    assert label({'Type':'Spam'}) is None
    assert label({'Type':'Phishing Simulation'}) is None
    assert label({'Type':'Valid'})==0
    assert threshold_at_fpr([0]*30+[1]*20,np.linspace(0,1,50),0.05)>0
    m=score_metrics([0,0,1,1],[.1,.8,.7,.9],.5)
    assert m['fp']==1 and m['tp']==2
    return 'PASS'

start=time.monotonic(); test=smoke(); paths={}; sources={}
for name in SOURCE:
    paths[name],sources[name]=acquire(name)
raw={k:load(p) for k,p in paths.items()}
orig=group_index(raw['original'])
assert len(orig)==3075, 'Source revision/row count changed'
counts=Counter(str(get(r,'Type')) for r in orig.values())
assert counts=={'Phishing':1090,'Spam':1000,'Valid':950,'Phishing Simulation':35}, 'Original class taxonomy changed'
valid={k:r for k,r in orig.items() if label(r) is not None and txt(r).strip()}
# Remove normalized-text label disagreements and identical text aliases across different parents.
text_to_groups=defaultdict(set);text_to_classes=defaultdict(set)
for k,r in valid.items():
    h=digest_text(txt(r));text_to_groups[h].add(k);text_to_classes[h].add(label(r))
excluded_keys={k for h,ids in text_to_groups.items() if len(ids)>1 or len(text_to_classes[h])>1 for k in ids}
valid={k:r for k,r in valid.items() if k not in excluded_keys}
assert len(valid)>300 and len(set(label(r) for r in valid.values()))==2

def group_split(keys):
    y=[label(valid[k]) for k in keys]
    train,tmp=train_test_split(keys,test_size=.3,random_state=SEED,stratify=y)
    val,test=train_test_split(tmp,test_size=.5,random_state=SEED,stratify=[label(valid[k]) for k in tmp])
    return train,val,test

keys=sorted(valid)
train_ids,val_ids,test_ids=group_split(keys)
assert not(set(train_ids)&set(val_ids) or set(train_ids)&set(test_ids) or set(val_ids)&set(test_ids))
# Remove samples in validation/test whose text also occurs in training; crossing groups already excluded above.
def dataset(ids):return [txt(valid[k]) for k in ids],np.array([label(valid[k]) for k in ids],dtype=int)
X_train,y_train=dataset(train_ids);X_val,y_val=dataset(val_ids);X_test,y_test=dataset(test_ids)
assert len(set(y_train))==len(set(y_val))==len(set(y_test))==2
# Require sufficient legitimate controls (strict gate, no cherry-picking split).
assert sum(y_val==0)>=20 and sum(y_test==0)>=20

# Rewrite files contain same No. identity; parent test only, never train on test parent rewrites.
rewrites={}; coverage={}
for provider in ('gpt4o','deepseek','randomapi'):
    idx=group_index(raw[provider]); assert len(idx)==3075
    selected=[]; suspicious=0
    for k in test_ids:
        r=idx.get(k)
        if r is None or not txt(r).strip():continue
        # Inconsistent labeled source type is excluded rather than silently inherited.
        source_type=get(r,'Type')
        if source_type is not None and source_type!=get(orig[k],'Type'):
            suspicious+=1;continue
        selected.append((k,txt(r),label(valid[k])))
    rewrites[provider]=selected
    coverage[provider]={'usable_pairs':len(selected),'type_conflicts_excluded':suspicious}
    assert len(selected)>100

# Leakage checks: block rewritten test text if it is identical to original training text.
train_norm={digest_text(x) for x in X_train}
for provider,rows in rewrites.items():
    overlap=sum(digest_text(x) in train_norm for _,x,_ in rows)
    coverage[provider]['identical_to_training_text']=overlap
    if overlap:rewrites[provider]=[(k,x,y) for k,x,y in rows if digest_text(x) not in train_norm]

vec=TfidfVectorizer(ngram_range=(1,2),min_df=2,max_df=.98,sublinear_tf=True,max_features=120000,strip_accents='unicode')
train_mat=vec.fit_transform(X_train); val_mat=vec.transform(X_val); test_mat=vec.transform(X_test)
models={
 'tfidf_logistic':LogisticRegression(max_iter=1500,class_weight='balanced',random_state=SEED),
 'tfidf_linear_svm_calibrated':CalibratedClassifierCV(LinearSVC(class_weight='balanced',random_state=SEED),method='sigmoid',cv=3)
}
results=[]; paired=[]
for model_name,model in models.items():
    model.fit(train_mat,y_train)
    pval=model.predict_proba(val_mat)[:,1]
    threshold=threshold_at_fpr(y_val,pval,.05)
    ptest=model.predict_proba(test_mat)[:,1]
    results.append({'model':model_name,'partition':'heldout_original','threshold_validation_5pct_fpr':threshold,**score_metrics(y_test,ptest,threshold)})
    baseline={k:float(score) for k,score in zip(test_ids,ptest)}
    for provider,rows in rewrites.items():
        ids=[k for k,_,_ in rows]; x=[x for _,x,_ in rows]; y=np.asarray([y for _,_,y in rows],dtype=int)
        scores=model.predict_proba(vec.transform(x))[:,1]
        results.append({'model':model_name,'partition':'heldout_rewrite_'+provider,'threshold_validation_5pct_fpr':threshold,**score_metrics(y,scores,threshold)})
        for cla,name in ((0,'legitimate'),(1,'phishing')):
            mask=(y==cla)
            if not mask.any():continue
            delta=[float(z-baseline[k]) for (k,_,yy),z in zip(rows,scores) if yy==cla]
            orig_wrong=[baseline[k]>=threshold for k,_,yy in rows if yy==cla]
            rewrite_wrong=[z>=threshold for (_,_,yy),z in zip(rows,scores) if yy==cla]
            if cla==1:
                orig_wrong=[not x for x in orig_wrong];rewrite_wrong=[not x for x in rewrite_wrong]
            paired.append({'model':model_name,'provider':provider,'target_class':name,'n':int(mask.sum()),
                'original_error_rate_on_same_parent_subset':float(np.mean(orig_wrong)),
                'rewrite_error_rate':float(np.mean(rewrite_wrong)),
                'rewrite_minus_original_error_rate':float(np.mean(rewrite_wrong)-np.mean(orig_wrong)),
                'mean_rewrite_minus_original_phishing_score':float(np.mean(delta))})

# Save aggregates only. No raw text, row-level scores, sender, URL or IDs in output.
summary={'release':RELEASE,'utc':datetime.now(timezone.utc).isoformat(),'status':'EXECUTED_SOURCE_ASSIGNED_LABEL_BENCHMARK',
 'runtime_seconds':round(time.monotonic()-start,2),'python':sys.version.split()[0],
 'platform':platform.platform(),'self_test':test,'source_integrity':sources,
 'source_class_counts':dict(counts),'excluded_ambiguous_or_duplicate_parent_ids':len(excluded_keys),
 'split_counts':{'train':len(train_ids),'validation':len(val_ids),'test':len(test_ids)},
 'split_label_counts':{z:dict(Counter(map(int,y))) for z,y in [('train',y_train),('val',y_val),('test',y_test)]},
 'source_rewrite_coverage':coverage,'models':[x for x in models],
 'limitations':['D2 labels are source-assigned, not independent gold','GPT-4o/DeepSeek outputs may be model-rewritten but intent not individually human verified',
 'This is matched-parent robustness, not independent unseen-corpus evaluation','RandomAPI is not a verified single-generator identifier',
 'No cross-language D4 comparisons here','Spam excluded from binary target','Source rights/privacy and publication reuse require separate assessment']}

def write_csv(name,rows):
    with (OUT/name).open('w',encoding='utf-8',newline='') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
(OUT/'summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
write_csv('metrics.csv',results);write_csv('paired_effects.csv',paired)
(OUT/'README.txt').write_text('N05 E01 safe aggregate outputs. Source-assigned labels, not human gold. No raw email included.\n',encoding='utf-8')
summary['output_sha256']={p.name:sha256(p) for p in sorted(OUT.iterdir()) if p.is_file()}
(OUT/'run_manifest.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
print('N05 completed; SAFE AGGREGATES:',OUT)
print('Split counts:',summary['split_counts'],'Excluded duplicated parents:',len(excluded_keys))
for row in paired:print(row['model'],row['provider'],row['target_class'],'n=',row['n'],'error delta=',round(row['rewrite_minus_original_error_rate'],4))


## Interpret safely

`paired_effects.csv` compares original vs rewritten emails within **the same held-out parent IDs** at thresholds frozen on the original validation partition. Negative values do not imply an improvement caused by AI in general; they describe this particular labeled dataset. Review class `Valid` is an inherited **source** annotation, not independently verified benign intent. `RandomAPI` is a mixed pipeline. No external D4 test is run in this notebook.

To share: download only `/kaggle/working/phishing_n05_safe/` metadata and tables. Never export `/kaggle/temp/phishing_n05_private/` or raw messages to a public repository. Commit a reviewed privacy-safe results ZIP and this output-stripped notebook to GitHub.


# N06 — Exact paired inference, confidence intervals, and selective reliability

This notebook keeps the complete frozen N05 pipeline and adds parent-paired error-discordance counts, exact McNemar tests, 10,000-bootstrap paired intervals, exact binomial confidence intervals and selective risk–coverage. **No new independent source or AI-authored benign dataset is introduced here.** Test-stage output contains only aggregate tables. The original N05 cells are retained for reproducibility; no test-set tuning or model changes.

In [ ]:
# N06 — journal statistics extension of frozen N05, same held-out IDs and thresholds.
# No private per-row information or raw text is written to /kaggle/working.
from scipy.stats import binomtest, beta
from math import isfinite
N06_OUT=ROOT/'phishing_n06_safe'
N06_OUT.mkdir(parents=True,exist_ok=True)
N_BOOT=10000
def paired_inference(original_error, rewritten_error, seed=2026, reps=N_BOOT):
    a=np.asarray(original_error,dtype=int)
    b=np.asarray(rewritten_error,dtype=int)
    if len(a)!=len(b) or not len(a) or not np.isin(a,[0,1]).all() or not np.isin(b,[0,1]).all():
        raise ValueError("Invalid paired arrays; nonbinary/empty/misaligned")
    n01=int(np.sum((a==0)&(b==1)))
    n10=int(np.sum((a==1)&(b==0)))
    delta=b-a
    rng=np.random.default_rng(seed)
    # Bootstrap parent IDs, not rows from other providers, independently.
    ix=rng.integers(0,len(a),size=(reps,len(a)))
    samples=delta[ix].mean(axis=1)
    lo,hi=np.quantile(samples,[.025,.975])
    p=float(binomtest(min(n01,n10),n01+n10,p=.5).pvalue) if n01+n10 else 1.0
    return {'n':int(len(a)), 'original_errors':int(a.sum()),'rewrite_errors':int(b.sum()),
            'n_original_correct_rewrite_wrong':n01,
            'n_original_wrong_rewrite_correct':n10,
            'absolute_error_delta':float(delta.mean()),'bootstrap_ci95_low':float(lo),
            'bootstrap_ci95_high':float(hi),'mcnemar_exact_p_two_sided':p,
            'bootstrap_iterations':reps}
def exact_binomial_ci(k,n,alpha=.05):
    if n<1 or k<0 or k>n:raise ValueError("bad count")
    return (0.0 if k==0 else float(beta.ppf(alpha/2,k,n-k+1)),
            1.0 if k==n else float(beta.ppf(1-alpha/2,k+1,n-k)))
def selective_metrics(y,s,t,coverage):
    y=np.asarray(y,dtype=int);s=np.asarray(s,dtype=float)
    assert len(y)==len(s) and all(isfinite(x) for x in s)
    # Confidence is distance from the validation-frozen operating threshold.
    # Ties are broken by deterministic row order; use only descriptive evaluation.
    n=len(y);count=max(1,int(np.floor(coverage*n)))
    chosen=np.argsort(-np.abs(s-t),kind='stable')[:count]
    pred=(s[chosen]>=t).astype(int)
    nonphish=(y[chosen]==0)
    return {'coverage':float(count/n),'accepted':int(count),
            'error_rate_accepted':float(np.mean(pred!=y[chosen])),
            'fpr_accepted':float(np.mean(pred[nonphish]==1)) if nonphish.any() else None,
            'legitimate_accepted':int(nonphish.sum())}
def statistical_smoke():
    x=paired_inference([0,0,1,1],[1,0,0,1],reps=500)
    assert x['n_original_correct_rewrite_wrong']==1 and x['n_original_wrong_rewrite_correct']==1
    assert x['absolute_error_delta']==0 and x['mcnemar_exact_p_two_sided']==1.0
    z=paired_inference([0,0,0],[0,0,0],reps=500)
    assert z['mcnemar_exact_p_two_sided']==1.0 and z['bootstrap_ci95_low']==0
    ci=exact_binomial_ci(0,30);assert ci[0]==0 and ci[1]>0
    s=selective_metrics([0,0,1,1],[.1,.2,.8,.9],.5,.5)
    assert s['accepted']==2 and s['error_rate_accepted']==0
statistical_smoke()
n06_pairs=[];n06_selective=[];n06_rate=[]
for name,model in models.items():
    pval=model.predict_proba(val_mat)[:,1]
    t=threshold_at_fpr(y_val,pval,.05)
    original_p=model.predict_proba(test_mat)[:,1]
    originals={k:float(p) for k,p in zip(test_ids,original_p)}
    # Check exact N05 confusion matrix for reproducibility (fail closed).
    n05r=next(r for r in results if r['model']==name and r['partition']=='heldout_original')
    assert score_metrics(y_test,original_p,t)['fp']==n05r['fp']
    for c in (0,1):
        ids=[k for k,y in zip(test_ids,y_test) if y==c]
        n=len(ids); k=sum((originals[i]>=t) if c==0 else (originals[i]<t) for i in ids)
        lo,hi=exact_binomial_ci(k,n)
        n06_rate.append({'model':name,'variant':'original','class':'legitimate' if c==0 else 'phishing',
                         'n':n,'errors':int(k),'rate':k/n,'exact_ci95_low':lo,'exact_ci95_high':hi})
    for fraction in (.5,.8,1.):
        n06_selective.append({'model':name,'variant':'original',**selective_metrics(y_test,original_p,t,fraction)})
    for provider,rows in rewrites.items():
        ids=[k for k,_,_ in rows]; x=[x for _,x,_ in rows]
        y=np.asarray([yy for _,_,yy in rows],dtype=int)
        p=model.predict_proba(vec.transform(x))[:,1]
        for c,cn in ((0,'legitimate'),(1,'phishing')):
            mask=(y==c); psub=p[mask]; ids_sub=[k for k,yy in zip(ids,y) if yy==c]
            old=np.array([int(originals[k]>=t) if c==0 else int(originals[k]<t) for k in ids_sub])
            new=np.array([int(z>=t) if c==0 else int(z<t) for z in psub])
            stat=paired_inference(old,new,seed=SEED)
            n06_pairs.append({'model':name,'provider':provider,'class':cn,**stat})
            low,high=exact_binomial_ci(int(new.sum()),len(new))
            n06_rate.append({'model':name,'variant':provider,'class':cn,'n':int(len(new)),
                             'errors':int(new.sum()),'rate':float(new.mean()),
                             'exact_ci95_low':low,'exact_ci95_high':high})
        for fraction in (.5,.8,1.):
            n06_selective.append({'model':name,'variant':provider,**selective_metrics(y,p,t,fraction)})
def save_n06_csv(filename,records):
    assert records and all(set(r)==set(records[0]) for r in records)
    with (N06_OUT/filename).open('w',newline='',encoding='utf-8') as fd:
        w=csv.DictWriter(fd,fieldnames=list(records[0]))
        w.writeheader();w.writerows(records)
save_n06_csv('paired_exact_tests.csv',n06_pairs)
save_n06_csv('error_rate_intervals.csv',n06_rate)
save_n06_csv('selective_risk_coverage.csv',n06_selective)
n06_report={
 'release':'N06-E02-PairedStats-v1.0-2026-11-02',
 'status':'LOCAL_SYNTHETIC_TEST_PASSED; REAL_RESULTS_ONLY_AFTER_KAGGLE_EXECUTION',
 'n06_synthetic_selftest':'PASS',
 'parent_group_reused_from':'N05-E01-v1.0-2026-11-02',
 'seed':SEED,'bootstrap_iterations':N_BOOT,
 'analysis':'exact McNemar + paired bootstrap by original parent ID; exact binomial error-rate CIs; selective risk-coverage',
 'data_label_caution':'source-assigned D2 labels; AI rewrite does not establish independent AI authorship or intent preservation',
 'inference_caution':'multiple exploratory comparisons not multiplicity-adjusted; small n=142 legitimate; no independent cross-corpus experiment',
 'files':{}}
for fn in ('paired_exact_tests.csv','error_rate_intervals.csv','selective_risk_coverage.csv'):
    n06_report['files'][fn]=sha256(N06_OUT/fn)
(N06_OUT/'run_manifest.json').write_text(json.dumps(n06_report,indent=2),encoding='utf-8')
print('N06 paired statistics complete; safe aggregate files:',N06_OUT)
for r in n06_pairs:
    if r['class']=='legitimate':
        print(r['model'],r['provider'],'discordance:',r['n_original_correct_rewrite_wrong'],r['n_original_wrong_rewrite_correct'],
              'delta:',round(r['absolute_error_delta'],4),'p:',r['mcnemar_exact_p_two_sided'])


## Interpretation protocol

An observed extra 1/142 false positive is a **descriptive change**, not evidence of an AI-authorship effect. Paired discordant transitions and exact two-sided McNemar p-values are required. Bootstrap intervals quantify sampling uncertainty over parent IDs but do not correct source-label errors, rewrite semantic changes or broader source bias. Results are exploratory: multiple providers and models are not familywise-adjusted. Independent corpus and genuinely AI-authored benign controls remain separate necessary studies.

# N07 — Train-only cross-validated classical model comparisons

Adds word-level and character-level TF-IDF, 1–3 grams, Naive Bayes and optional LightGBM. Each pipeline fits its vocabulary **inside** three stratified training folds. Per-family configuration is selected only from inner-CV average precision. Decision threshold is then selected on original validation only. Frozen N05 and N06 results appear above. Held-out test has already been examined during N05, so **all new comparisons are exploratory until replicated on a new independent holdout**. No claim of transformer SOTA or Q1 readiness is made.

In [ ]:
# N07 — additional classical models, all tuning inside original training partition only.
# Original N05 baselines and N06 analyses above stay frozen.
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.base import clone
import sklearn
N07_OUT=ROOT/'phishing_n07_safe'
N07_OUT.mkdir(parents=True,exist_ok=True)
inner_cv=StratifiedKFold(n_splits=3,shuffle=True,random_state=SEED)
def make_candidates():
    word=dict(strip_accents='unicode',min_df=2,max_df=.98,sublinear_tf=True,max_features=80000)
    char=dict(analyzer='char_wb',ngram_range=(3,5),min_df=3,max_features=100000,sublinear_tf=True)
    candidates={}
    for c in (.3,1.,3.):
        candidates['word_lr_C'+str(c)]=Pipeline([('tfidf',TfidfVectorizer(ngram_range=(1,2),**word)),
                 ('clf',LogisticRegression(C=c,max_iter=1500,class_weight='balanced',random_state=SEED))])
    for alpha in (.1,1.,3.):
        candidates['word_nb_alpha'+str(alpha)]=Pipeline([('tfidf',TfidfVectorizer(ngram_range=(1,2),**word)),
                    ('clf',MultinomialNB(alpha=alpha))])
    for c in (.3,1.,3.):
        candidates['char_lr_C'+str(c)]=Pipeline([('tfidf',TfidfVectorizer(**char)),
                    ('clf',LogisticRegression(C=c,max_iter=1500,class_weight='balanced',random_state=SEED))])
    for c in (.3,1.):
        candidates['word_trigram_lr_C'+str(c)]=Pipeline([
                    ('tfidf',TfidfVectorizer(ngram_range=(1,3),**word)),
                    ('clf',LogisticRegression(C=c,max_iter=1500,class_weight='balanced',random_state=SEED))])
    try:
        import lightgbm as lgb
        for leaves in (15,31):
            candidates['word_lgb_leaves'+str(leaves)]=Pipeline([
                ('tfidf',TfidfVectorizer(ngram_range=(1,2),**word)),
                ('clf',lgb.LGBMClassifier(n_estimators=160,learning_rate=.05,num_leaves=leaves,
                    max_depth=-1,colsample_bytree=.8,subsample=.9,reg_lambda=1.,
                    verbosity=-1,n_jobs=2,random_state=SEED))])
        lgb_status='available'
    except ImportError:
        lgb_status='not-installed: SKIPPED (no implicit network install)'
    return candidates,lgb_status
candidate_pipes,lgb_status=make_candidates()
FAMILIES=['word_lr','word_nb','char_lr','word_trigram_lr']
if lgb_status=='available':FAMILIES.append('word_lgb')
def family(name):
    return next(f for f in FAMILIES if name.startswith(f+'_'))
def n07_smoke():
    # Pipeline must contain TF-IDF inside each fold, preventing vocab leakage.
    for estimator in candidate_pipes.values():
        assert isinstance(estimator,Pipeline) and estimator.steps[0][0]=='tfidf'
    toy=['urgent payment login','meeting notes agenda','verify bank password','team lunch tomorrow',
         'security sign in','hello event schedule']*3
    yy=np.array([1,0,1,0,1,0]*3)
    mdl=Pipeline([('tfidf',TfidfVectorizer(ngram_range=(1,2),min_df=1)),
                  ('clf',LogisticRegression(max_iter=1000))])
    z=cross_val_score(mdl,toy,yy,cv=inner_cv,scoring='average_precision')
    assert len(z)==3 and np.isfinite(z).all()
n07_smoke()
cv_rows=[]
for candidate,pipe in candidate_pipes.items():
    scores=cross_val_score(pipe,X_train,y_train,cv=inner_cv,scoring='average_precision',n_jobs=1)
    if not np.isfinite(scores).all():raise RuntimeError('Nonfinite inner-CV score: '+candidate)
    cv_rows.append({'family':family(candidate),'configuration':candidate,
                    'inner_cv_ap_mean':float(scores.mean()),
                    'inner_cv_ap_sd':float(scores.std(ddof=1)),
                    'n_folds':3})
selections={}
for fam in FAMILIES:
    scores=[r for r in cv_rows if r['family']==fam]
    if scores:
        selections[fam]=sorted(scores,key=lambda x:(-x['inner_cv_ap_mean'],x['configuration']))[0]['configuration']
assert selections and len(set(selections.values()))==len(selections)
n07_metrics=[];n07_effects=[]
for fam,selected in selections.items():
    model=clone(candidate_pipes[selected])
    model.fit(X_train,y_train)
    val_p=model.predict_proba(X_val)[:,1]
    threshold=threshold_at_fpr(y_val,val_p,.05)
    original_p=model.predict_proba(X_test)[:,1]
    n07_metrics.append({'family':fam,'configuration':selected,'variant':'original',
                        'threshold':float(threshold),**score_metrics(y_test,original_p,threshold)})
    base={k:float(p) for k,p in zip(test_ids,original_p)}
    for provider,rows in rewrites.items():
        ids=[k for k,_,_ in rows]
        yy=np.asarray([y for _,_,y in rows],dtype=int)
        prob=model.predict_proba([x for _,x,_ in rows])[:,1]
        n07_metrics.append({'family':fam,'configuration':selected,'variant':provider,
                            'threshold':float(threshold),**score_metrics(yy,prob,threshold)})
        for labelid,classname in ((0,'legitimate'),(1,'phishing')):
            selectedidx=np.where(yy==labelid)[0]
            old=np.array([int(base[ids[i]]>=threshold) if labelid==0 else int(base[ids[i]]<threshold)
                         for i in selectedidx],dtype=int)
            new=np.array([int(prob[i]>=threshold) if labelid==0 else int(prob[i]<threshold)
                         for i in selectedidx],dtype=int)
            stat=paired_inference(old,new,seed=SEED,reps=10000)
            n07_effects.append({'family':fam,'provider':provider,'class':classname,**stat})
save_n06_csv('not_used.csv',[{'placeholder':1}]) if False else None
def save_n07_csv(name,records):
    assert records and all(set(r)==set(records[0]) for r in records)
    with (N07_OUT/name).open('w',encoding='utf-8',newline='') as fd:
        writer=csv.DictWriter(fd,fieldnames=list(records[0]))
        writer.writeheader();writer.writerows(records)
save_n07_csv('inner_cv_scores.csv',cv_rows)
save_n07_csv('model_comparison.csv',n07_metrics)
save_n07_csv('paired_effects.csv',n07_effects)
info={'release':'N07-E03-ClassicalCV-v1.0-2026-11-02','status':'KAGGLE_REAL_DATA_COMPLETED',
      'selftest':'PASS','source':'same D2 pinned source as N05; no external source',
      'inner_cv':'3-fold StratifiedKFold on N05 train originals; TF-IDF fitted per fold in sklearn Pipeline',
      'model_selection':'highest mean inner-CV average precision per family, tie-break lexical name',
      'selection':selections,'lightgbm':lgb_status,'sklearn':sklearn.__version__,
      'reported_variants':['original']+list(rewrites),
      'limitations':['Same already observed N05 held-out test; not a newly sealed external test',
                     'Not independent AI-authored legitimate mail; source-annotated original and rewrites',
                     'CV is train-only, but published test remains exploratory for research questions shaped by N05',
                     'Calibration not separately fitted for new model families',
                     'Multiple-comparison p-values are exploratory and not adjusted'],'files':{}}
for fn in ['inner_cv_scores.csv','model_comparison.csv','paired_effects.csv']:
    info['files'][fn]=sha256(N07_OUT/fn)
(N07_OUT/'run_manifest.json').write_text(json.dumps(info,indent=2),encoding='utf-8')
print('N07 classical model comparison completed, privacy-safe folder:',N07_OUT)
print('Selected configurations:',selections)
print('Optional LightGBM:',lgb_status)


## Research interpretation

The key endpoint is legitimate original and rewritten false-positive rate at a validation-selected threshold, conditional on held-out phishing recall. PR-AUC, Brier and class-specific paired statistics are supporting outcomes. Inner-CV optimization is permitted; retuning models on the previously seen test is not. Add an independent, source-clean held-out dataset and a fine-tuned transformer before drawing journal-grade generalization conclusions.

# N08 — External AI-authored legitimate email evaluation (candidate)

Independently authored synthetic source **E-PhishLLM (ACM AISec 2025)**, distinct from D2 same-parent rewrites. The official dataset Git blob hash is enforced, with fail-closed schema checks and English-only primary evaluation. Existing D2 models/thresholds are reused unchanged; all raw text remains in a private directory. Direct AI-vs-human **causal** interpretation is disallowed because corpora differ in topic, style and generator. Source-level near-duplicate matching and license/rights review remain necessary before making strong independent generalization claims.

In [ ]:
# N08 — Immutable-blob-gated external E-PhishLLM corpus, zero external-data tuning.
# Expected Git blob obtained from official pajola/e-phishGen/ephishLLM.json.
EXT_FILE='ephishLLM.json'
EXT_GIT_BLOB_SHA='7a0e74ecebc1f3f5b21f42deccfb716b14bba46b'
EXT_URL='https://raw.githubusercontent.com/pajola/e-phishGen/main/ephishLLM.json'
N08_OUT=ROOT/'phishing_n08_safe'; N08_OUT.mkdir(parents=True,exist_ok=True)
EXT_PRIVATE=PRIV/'ephishLLM_external'
EXT_PRIVATE.mkdir(parents=True,exist_ok=True)
def external_file():
    candidates=list(Path('/kaggle/input').rglob(EXT_FILE)) if Path('/kaggle/input').exists() else []
    candidates.append(EXT_PRIVATE/EXT_FILE)
    for p in candidates:
        if p.is_file() and git_blob(p)==EXT_GIT_BLOB_SHA:
            return p,'verified_private_cache'
    p=EXT_PRIVATE/EXT_FILE; tmp=EXT_PRIVATE/(EXT_FILE+'.part')
    try:
        with urllib.request.urlopen(EXT_URL,timeout=120) as r, tmp.open('wb') as fd:
            amount=0
            for block in iter(lambda:r.read(1<<20),b''):
                amount+=len(block)
                if amount>30_000_000:raise ValueError('Unexpectedly large external source')
                fd.write(block)
        if git_blob(tmp)!=EXT_GIT_BLOB_SHA:
            raise ValueError('External official E-PhishLLM content changed: PIN MISMATCH')
        tmp.replace(p)
        return p,'blob_verified_download'
    finally:
        if tmp.exists():tmp.unlink()
def check_external_schema(data):
    if not isinstance(data,list) or len(data)!=16616 or any(not isinstance(r,dict) for r in data):
        raise ValueError('External dataset unexpected length/schema; stop and audit')
    observed=Counter()
    for r in data:
        cls=r.get('type')
        if cls not in (0,1,'0','1'):raise ValueError('Invalid external phishing type')
        if not ('Subject' in r and 'Body' in r and 'Language' in r):
            raise ValueError('Required E-PhishLLM fields missing')
        observed[str(cls)]+=1
    if min(observed['0'],observed['1'])<1000:raise ValueError('Insufficient external class supports')
    return observed
def n08_smoke():
    t=[{'Subject':'alpha','Body':'body','type':0,'Language':'en'},
       {'Subject':'beta','Body':'body','type':1,'Language':'en'}]
    try:check_external_schema(t)
    except ValueError:pass
    else:raise AssertionError('invalid external size accepted')
    assert git_blob is not None
n08_smoke()
ext_path,ext_source_origin=external_file()
with ext_path.open(encoding='utf-8') as fd: ext_data=json.load(fd)
ext_counts=check_external_schema(ext_data)
langs=Counter(str(r['Language']).strip().lower() for r in ext_data)
# Register every D2 original and LLM rewrite text, to exclude cross-family exact reuse.
d2_reference={digest_text(txt(r)) for corpus in raw.values() for r in corpus}
ext_records=[]; seen_external=set(); empty=nonenglish=overlap=dupes=0
for row in ext_data:
    lang=str(row['Language']).strip().lower()
    if lang not in ('en','eng','english'):
        nonenglish+=1;continue
    body=str(row.get('Body') or '');subject=str(row.get('Subject') or '')
    x=subject+'\n'+body
    if not norm(x):empty+=1;continue
    h=digest_text(x)
    if h in d2_reference:overlap+=1;continue
    if h in seen_external:dupes+=1;continue
    seen_external.add(h)
    ext_records.append((x,int(row['type'])))
if len(ext_records)<100 or len({y for _,y in ext_records})!=2:
    raise ValueError('No adequate decontaminated English external support')
external_x=[x for x,y in ext_records]
external_y=np.asarray([y for x,y in ext_records],dtype=int)
# N05 frozen models were fit on training originals; N07 best configs are from train-only inner CV.
external_models={}
for name,model in models.items():
    external_models['n05_'+name]=(model,threshold_at_fpr(y_val,model.predict_proba(val_mat)[:,1],.05))
for fam,chosen in selections.items():
    m=clone(candidate_pipes[chosen]);m.fit(X_train,y_train)
    external_models['n07_'+fam]=(m,threshold_at_fpr(y_val,m.predict_proba(X_val)[:,1],.05))
ext_stats=[];ext_fpr=[]
for name,(mdl,t) in external_models.items():
    # N05 uses separate frozen vectorizer, N07 Pipeline includes internal TF-IDF.
    if name.startswith('n05_'):
        probs=mdl.predict_proba(vec.transform(external_x))[:,1]
    else:
        probs=mdl.predict_proba(external_x)[:,1]
    ext_stats.append({'model':name,'partition':'external_ephishllm_english',
                      'threshold_from_d2_validation':float(t),
                      **score_metrics(external_y,probs,t)})
    for cl,cn in ((0,'AI_authored_synthetic_legitimate'),(1,'AI_authored_synthetic_phishing')):
        mask=external_y==cl
        wrong=(probs[mask]>=t) if cl==0 else (probs[mask]<t)
        lo,hi=exact_binomial_ci(int(wrong.sum()),int(mask.sum()))
        ext_fpr.append({'model':name,'class':cn,'n':int(mask.sum()),
                        'errors':int(wrong.sum()),'rate':float(wrong.mean()),
                        'exact_ci95_low':lo,'exact_ci95_high':hi})
save_n07_csv('skip.csv',[{'unused':0}]) if False else None
def save_n08_csv(name,records):
    assert records and all(set(r)==set(records[0]) for r in records)
    with (N08_OUT/name).open('w',newline='',encoding='utf-8') as fd:
        w=csv.DictWriter(fd,fieldnames=list(records[0]));w.writeheader();w.writerows(records)
save_n08_csv('external_model_metrics.csv',ext_stats)
save_n08_csv('external_class_error_intervals.csv',ext_fpr)
n08_summary={
 'release':'N08-E04-ExternalAIAuthored-v1.0-2026-11-02',
 'status':'KAGGLE_REAL_DATA_COMPLETED','source':'pajola/e-phishGen/ephishLLM.json',
 'source_published_paper':'https://doi.org/10.1145/3733799.3762967',
 'pinned_git_blob_sha1':EXT_GIT_BLOB_SHA,'sha256_downloaded_bytes':sha256(ext_path),
 'source_origin':ext_source_origin,'source_total':len(ext_data),'source_class_counts':dict(ext_counts),
 'source_language_counts':dict(langs),'english_unique_decontaminated':len(ext_records),
 'excluded_non_english':nonenglish,'excluded_empty':empty,
 'excluded_exact_cross_source':overlap,'excluded_internal_exact_duplicate':dupes,
 'evaluation':'original D2 train-fitted classifiers with D2 validation-frozen thresholds; no external-data fitting',
 'limitations':['Synthetic LLM-authored benign != verified human benign; no causal AI style comparison',
                'English-only main evaluation; other language samples excluded deliberately',
                'No measured near-duplicate clustering beyond exact normalized text',
                'Datasets may differ in topic, message conventions, synthetic artifacts and label policies',
                'N05 test already seen before N07 model family planning; N08 external outcomes must not drive further tuning',
                'Dataset license and raw-message publication rights not established here'],
 'files':{}}
for fn in ['external_model_metrics.csv','external_class_error_intervals.csv']:
    n08_summary['files'][fn]=sha256(N08_OUT/fn)
(N08_OUT/'run_manifest.json').write_text(json.dumps(n08_summary,indent=2),encoding='utf-8')
print('N08 external source evaluated (no raw text exported):',N08_OUT)
print('English support:',len(ext_records),'source languages:',dict(langs),'cross-exact overlap:',overlap)


## Research gate

Report the exact number of English legitimate and phishing samples, source-language breakdown, excluded cross-source normalized duplicates, false-positive rate, phishing recall, Brier/ECE and confidence intervals. Evaluate the source **once without changing training or decision thresholds**. This source was already used by its creators for benchmarking; using it is not independently novel. For journal novelty, evaluate a new security-evidence-aware mechanism against these frozen results and separately document legitimate email author provenance.

# N09 — Ablation: lexical style versus security-cue proxies

Compares three families trained on exactly N05 train originals: 10 stylistic aggregate features, 12 security-related **lexical proxies**, and a TF-IDF + evidence-proxy fusion selected via train-only CV. Evaluation uses N05 parent-held-out originals/rewrites and N08 decontaminated external E-PhishLLM at thresholds selected only on D2 validation. Important: presence of words about passwords, links or bank accounts is **not actual evidence of malicious intent**. This experiment cannot establish causal feature dependence, a novel architecture, or security label preservation; it provides a first controlled ablation and hypothesis screening.

In [ ]:
# N09 — content-vs-style ablation and evidence-proxy fusion, no external-data tuning.
# NOTE: text-derived signals are PROXIES, not verified identity/URL/attachment evidence.
from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import FunctionTransformer,StandardScaler
N09_OUT=ROOT/'phishing_n09_safe';N09_OUT.mkdir(parents=True,exist_ok=True)
def evidence_proxy_features(messages):
    out=[]
    for message in messages:
        s=str(message).lower()
        words=re.findall(r'\b\w+\b',s)
        out.append([
            int(bool(re.search(r'https?://|www\.|[a-z0-9.-]+\.(?:com|net|org|info|co)\b',s))),
            int('click' in s),int(bool(re.search(r'password|passcode|credentials|otp|two.factor',s))),
            int(bool(re.search(r'login|log.in|sign.in|account access',s))),
            int(bool(re.search(r'verify|confirm|validate',s))),
            int(bool(re.search(r'payment|invoice|bank|transfer|refund|billing',s))),
            int(bool(re.search(r'urgent|immediately|within 24|expires|deadline',s))),
            int(bool(re.search(r'attachment|attached|enclosed',s))),
            int(bool(re.search(r'account (?:suspended|locked)|security alert|unusual activity',s))),
            int(bool(re.search(r'unsubscribe|newsletter|promotion',s))),
            int(bool(re.search(r'dear (?:customer|user|account)',s))),
            float(min(len(words),2000)/2000.)
        ])
    return np.asarray(out,dtype=float)
def stylistic_features(messages):
    out=[]
    for message in messages:
        s=str(message);words=re.findall(r'\b\w+\b',s)
        n=max(len(words),1)
        out.append([
            min(len(words),2000)/2000.,
            min(sum(map(len,words))/n,20)/20.,
            min(s.count('!'),20)/20.,
            min(s.count('?'),20)/20.,
            min(s.count('.'),40)/40.,
            min(s.count('\n'),40)/40.,
            min(sum(c.isupper() for c in s)/max(len(s),1),1.),
            int(bool(re.search(r'\bplease\b|\bkindly\b|\bregards\b|\bsincerely\b',s.lower()))),
            int(bool(re.search(r'\bhi\b|\bhello\b|\bdear\b',s.lower()))),
            min(len(s),10000)/10000.,
        ])
    return np.asarray(out,dtype=float)
def simple_features(fn):
    return Pipeline([('features',FunctionTransformer(fn,validate=False)),
                     ('scale',StandardScaler()),
                     ('clf',LogisticRegression(C=1.,class_weight='balanced',max_iter=1500,random_state=SEED))])
def fusion_pipeline(weight):
    return Pipeline([
     ('features',FeatureUnion([
        ('text',TfidfVectorizer(ngram_range=(1,2),min_df=2,max_df=.98,
                sublinear_tf=True,max_features=80000,strip_accents='unicode')),
        ('evidence',Pipeline([
            ('extract',FunctionTransformer(evidence_proxy_features,validate=False)),
            ('scale',StandardScaler(with_mean=False))]))
      ],transformer_weights={'text':1.0,'evidence':weight})),
      ('clf',LogisticRegression(C=1.,class_weight='balanced',max_iter=1500,random_state=SEED))
     ])
def ablation_smoke():
    toy=['Dear team, please review the meeting notes.','URGENT: verify password at https://example.com']
    s=stylistic_features(toy);e=evidence_proxy_features(toy)
    assert s.shape==(2,10) and e.shape==(2,12)
    assert np.isfinite(s).all() and np.isfinite(e).all()
    assert e[1,0]==1 and e[1,2]==1
ablation_smoke()
n09_candidates={'style_only':simple_features(stylistic_features),
                'evidence_proxy_only':simple_features(evidence_proxy_features),
                'fusion_0.5':fusion_pipeline(.5),
                'fusion_1.0':fusion_pipeline(1.),
                'fusion_2.0':fusion_pipeline(2.)}
n09_cv=[]
for name,pipe in n09_candidates.items():
    score=cross_val_score(pipe,X_train,y_train,cv=inner_cv,scoring='average_precision',n_jobs=1)
    n09_cv.append({'configuration':name,'inner_cv_ap_mean':float(score.mean()),
                   'inner_cv_ap_sd':float(score.std(ddof=1)),'n_folds':3})
fusion_choice=sorted([x for x in n09_cv if x['configuration'].startswith('fusion_')],
                     key=lambda r:(-r['inner_cv_ap_mean'],r['configuration']))[0]['configuration']
selected_n09={'style_only':'style_only','evidence_proxy_only':'evidence_proxy_only',
              'evidence_fusion':fusion_choice}
n09_eval=[];n09_external=[];n09_paired=[]
for group,configuration in selected_n09.items():
    model=clone(n09_candidates[configuration])
    model.fit(X_train,y_train)
    thr=threshold_at_fpr(y_val,model.predict_proba(X_val)[:,1],.05)
    p0=model.predict_proba(X_test)[:,1]
    n09_eval.append({'model':group,'configuration':configuration,'variant':'original',
                     'threshold_from_validation':float(thr),**score_metrics(y_test,p0,thr)})
    origscore={k:float(p) for k,p in zip(test_ids,p0)}
    for provider,rows in rewrites.items():
        ids=[k for k,_,_ in rows];yy=np.asarray([y for _,_,y in rows],dtype=int)
        p=model.predict_proba([x for _,x,_ in rows])[:,1]
        n09_eval.append({'model':group,'configuration':configuration,'variant':provider,
                         'threshold_from_validation':float(thr),**score_metrics(yy,p,thr)})
        mask=yy==0
        old=np.asarray([int(origscore[k]>=thr) for k,yyi in zip(ids,yy) if yyi==0],dtype=int)
        new=np.asarray([int(v>=thr) for v in p[mask]],dtype=int)
        n09_paired.append({'model':group,'provider':provider,
                           **paired_inference(old,new,seed=SEED,reps=10000)})
    pe=model.predict_proba(external_x)[:,1]
    n09_external.append({'model':group,'configuration':configuration,'variant':'ephishllm_english',
                         'threshold_from_validation':float(thr),**score_metrics(external_y,pe,thr)})
save_n08_csv('unused.csv',[{'none':0}]) if False else None
def save_n09(name,records):
    with (N09_OUT/name).open('w',newline='',encoding='utf-8') as fd:
        w=csv.DictWriter(fd,fieldnames=list(records[0]))
        w.writeheader();w.writerows(records)
save_n09('inner_cv_ablation.csv',n09_cv)
save_n09('d2_ablation.csv',n09_eval)
save_n09('external_ablation.csv',n09_external)
save_n09('paired_legitimate_ablation.csv',n09_paired)
n09_report={'release':'N09-E05-EvidenceStyleAblation-v1.0-2026-11-02',
            'status':'KAGGLE_REAL_DATA_COMPLETED','selftest':'PASS',
            'training':'D2 original train IDs from N05 only; feature selection uses inner train CV; threshold uses D2 validation',
            'selected_configurations':selected_n09,
            'feature_warning':'Hand-coded token patterns are lexical evidence PROXIES, not independently verified security facts.',
            'novelty_warning':'Security-evidence fusion is a candidate method, NOT proven novel or superior until a controlled external outcome.',
            'external_warning':'E-PhishLLM synthetic external source has topic/generator/language confounds; not causal AI-bias data.',
            'files':{fn:sha256(N09_OUT/fn) for fn in
                     ('inner_cv_ablation.csv','d2_ablation.csv','external_ablation.csv',
                      'paired_legitimate_ablation.csv')}}
(N09_OUT/'run_manifest.json').write_text(json.dumps(n09_report,indent=2),encoding='utf-8')
print('N09 ablation completed:',N09_OUT,'chosen:',selected_n09)


## Claim gate

Publish as ablation only. A new 'security-aware' classifier is worth proposing only if source-clean external legitimate false alarms improve at a prespecified minimum phishing recall without worsening other strata. Compare against the same word TF-IDF classifier with identical training and validation protocol. Favor negative findings over post-hoc storytelling. Independently verified URL/header/domain metadata and source-level label review would strengthen a later causal/semantic interpretation.

# N10 — Privacy-safe publication-draft figures and result tables

Generates 4 figures (PNG 300 dpi + vector PDF), 5 aggregate result tables and a content-hashed run manifest. Runs after N05–N09 in **one cumulative Kaggle notebook**. This is a scientific reporting aid, not an automatic claim of Q1 novelty or causal AI-authorship bias. Every figure carries dataset/interpretation limitations; negative results should remain visible.

In [ ]:
# N10 — publication-draft figure/table compiler. All outputs aggregate-only.
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
N10_OUT=ROOT/'phishing_n10_safe';N10_OUT.mkdir(parents=True,exist_ok=True)
def save_figure(fig,stem):
    fig.savefig(N10_OUT/(stem+'.png'),dpi=300,bbox_inches='tight')
    fig.savefig(N10_OUT/(stem+'.pdf'),bbox_inches='tight')
    plt.close(fig)
def style_plot():
    plt.rcParams.update({'font.size':10,'axes.spines.top':False,
                         'axes.spines.right':False,'pdf.fonttype':42,
                         'figure.figsize':(8,4.5)})
style_plot()
# Figure 1 — original vs same-parent LLM rewrite FPR; primary N05 baselines.
fig,ax=plt.subplots()
providers=['heldout_original','heldout_rewrite_gpt4o','heldout_rewrite_deepseek','heldout_rewrite_randomapi']
names=['Original','GPT-4o','DeepSeek','RandomAPI']
xx=np.arange(len(providers));width=.34
for i,model in enumerate(models):
    ys=[100*next(r for r in results if r['model']==model and r['partition']==p)['fpr'] for p in providers]
    ax.bar(xx+(i-.5)*width,ys,width,label=model.replace('tfidf_',''),alpha=.85)
ax.set_xticks(xx,names);ax.set_ylabel('False-positive rate on source-labeled legitimate (%)')
ax.set_title('D2 matched-parent held-out email classification (N05)')
ax.legend(fontsize=8);ax.set_ylim(bottom=0)
save_figure(fig,'fig01_d2_legitimate_fpr')
# Figure 2 — exact paired delta and bootstrap intervals. CI intervals are two-sided.
leg=[r for r in n06_pairs if r['class']=='legitimate']
fig,ax=plt.subplots(figsize=(9,5))
yy=np.arange(len(leg))
effects=np.array([r['absolute_error_delta']*100 for r in leg])
lo=np.array([r['bootstrap_ci95_low']*100 for r in leg])
hi=np.array([r['bootstrap_ci95_high']*100 for r in leg])
ax.errorbar(effects,yy,xerr=np.vstack((np.maximum(0,effects-lo),np.maximum(0,hi-effects))),
            fmt='o',capsize=3,markersize=5)
ax.axvline(0,linestyle='--',linewidth=1,color='gray')
ax.set_yticks(yy,[r['model'].replace('tfidf_','')+' / '+r['provider'] for r in leg],fontsize=8)
ax.invert_yaxis();ax.set_xlabel('Rewrite − original false-positive rate (percentage points)')
ax.set_title('Parent-paired descriptive effects, 95% bootstrap CI (N06)')
save_figure(fig,'fig02_paired_legitimate_delta')
# Figure 3 — E-PhishLLM external FPR/recall under validation-fixed thresholds.
all_external=[{'model':r['model'],**r} for r in ext_stats]+[
    {'model':'n09_'+r['model'],**r} for r in n09_external]
fig,ax=plt.subplots(figsize=(7,5))
for item in all_external:
    if item['fpr'] is None or item['recall'] is None:continue
    ax.scatter(100*item['fpr'],100*item['recall'],s=40,alpha=.8)
    ax.annotate(item['model'].replace('tfidf_',''),(100*item['fpr'],100*item['recall']),
                xytext=(3,3),textcoords='offset points',fontsize=6)
ax.set_xlabel('External synthetic legitimate false-positive rate (%)')
ax.set_ylabel('External synthetic phishing recall (%)')
ax.set_title('E-PhishLLM English, frozen D2-trained thresholds')
save_figure(fig,'fig03_external_fpr_recall')
# Figure 4 — accepted error versus retained fraction (confidence from threshold distance).
fig,ax=plt.subplots(figsize=(7,5))
for model in models:
    subset=sorted((r for r in n06_selective if r['model']==model and r['variant']=='original'),
                  key=lambda z:z['coverage'])
    ax.plot([100*r['coverage'] for r in subset],
            [100*r['error_rate_accepted'] for r in subset],marker='o',label=model.replace('tfidf_',''))
ax.set_xlabel('Retained/automatically decided emails (%)')
ax.set_ylabel('Misclassification among retained (%)')
ax.set_title('Exploratory selective prediction on D2 held-out originals')
ax.legend(fontsize=8)
save_figure(fig,'fig04_selective_risk_coverage')
# Tables for manuscript construction, no raw records.
tables={
 'table01_n05_original_and_rewrites.csv':results,
 'table02_paired_exact_tests.csv':n06_pairs,
 'table03_n07_cv_selected.csv':n07_metrics,
 'table04_external_model_comparison.csv':ext_stats,
 'table05_style_evidence_ablation.csv':n09_external,
}
for name,rows in tables.items():
    with (N10_OUT/name).open('w',newline='',encoding='utf-8') as fd:
        w=csv.DictWriter(fd,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
figure_files=[p for p in N10_OUT.iterdir() if p.suffix in ('.png','.pdf')]
assert len(figure_files)==8 and all(p.stat().st_size>1000 for p in figure_files)
n10_manifest={'release':'N10-E06-Figures-v1.0-2026-11-02',
    'status':'KAGGLE_REAL_DATA_COMPLETED',
    'source_datasets':['HarmJ0y D2','E-PhishLLM published synthetic corpus'],
    'scope':'Aggregate-only descriptive tables and figures; draft manuscript material, not publication claim',
    'scientific_warnings':['N05 and later D2 test already explored; not a fresh holdout',
      'E-PhishLLM is a synthetic AI-authored corpus and differs in theme, language and generator',
      'None of these figures by itself establishes a new contribution or causal AI-style discrimination',
      'Figures generated before independent human pair-label review and source-rights gate'],
    'outputs':{p.name:sha256(p) for p in sorted(N10_OUT.iterdir()) if p.is_file()}}
(N10_OUT/'run_manifest.json').write_text(json.dumps(n10_manifest,indent=2),encoding='utf-8')
print('N10 publication-draft tables+plots generated:',N10_OUT)
print('4 PNG and 4 PDF figures; 5 tables; manifest. No raw emails published.')


## Publication readiness is not automatic

Before journal submission verify: immutable corpus pins, rights, source/label semantics, near-duplicate and campaign lineage, genuinely independent test, human-reviewed subset for security meaning, matched AI/human benign controls, source and prevalence confounds, adjusted hypothesis tests and code-release privacy. Q1 is a target journal quartile to verify at submission time, not an outcome guaranteed by this notebook.

# N11 — Rewrite augmentation intervention, source-safe exploratory study

**Novelty hypothesis, not a proven novel method.** Test whether injecting legitimate LLM rewrites into the *training-parent groups only* reduces external benign false positives while retaining phishing recall. Compare four fixed, same-capacity TF-IDF Logistic Regression strategies: originals only; benign rewrites only; all rewrites; GPT-4o rewrites. Every group contributes weight 1 in aggregate to avoid amplifying duplicated parent data; train samples are decontaminated against validation/test originals and held-out rewrites. Thresholds remain frozen from original validation. **The external E-PhishLLM data has already been seen in prior work; these new experiments are exploratory.**

In [ ]:
# N11 — Parent-safe benign-rewrite augmentation intervention (exploratory).
# SOURCE D2 label inheritance is not independently meaning-validated.
# All hyperparameters predetermined; external dataset used only for evaluation.
from sklearn.metrics import roc_curve
N11_OUT=ROOT/'phishing_n11_safe';N11_OUT.mkdir(parents=True,exist_ok=True)
TRAIN_SET=set(train_ids);VALID_SET=set(val_ids);TEST_SET=set(test_ids)
assert not (TRAIN_SET&VALID_SET or TRAIN_SET&TEST_SET or VALID_SET&TEST_SET)
def n11_aug_records():
    heldout_hash=set(digest_text(x) for x in X_val+X_test)
    # Keep ALL original test-provider messages out of augmented training if text collision.
    for provider,subset in rewrites.items():
        heldout_hash.update(digest_text(t) for _,t,_ in subset)
    seen=set(digest_text(x) for x in X_train)
    aug=[]; drops=Counter()
    for provider in ('gpt4o','deepseek','randomapi'):
        indexes=group_index(raw[provider])
        assert len(indexes)==len(orig)
        for pid in sorted(TRAIN_SET):
            row=indexes.get(pid)
            if row is None or get(row,'Type')!=get(valid[pid],'Type'):
                drops['source_type_or_missing']+=1;continue
            textval=txt(row)
            if not textval.strip():
                drops['empty']+=1;continue
            hx=digest_text(textval)
            if hx in heldout_hash:
                drops['colliding_with_heldout']+=1;continue
            if hx in seen:
                drops['identical_or_duplicate_training']+=1;continue
            seen.add(hx)
            aug.append((pid,provider,textval,int(label(valid[pid]))))
    assert aug and len(aug)>1000
    assert all(x[0] in TRAIN_SET and x[0] not in TEST_SET for x in aug)
    return aug,drops
AUG_ROWS,AUG_DROPS=n11_aug_records()
def augmentation_smoke():
    assert not any(pid in TEST_SET or pid in VALID_SET for pid,_,_,_ in AUG_ROWS)
    assert all(digest_text(textval) not in {digest_text(x) for x in X_test}
               for pid,_,textval,_ in AUG_ROWS[:20])
augmentation_smoke()
# Keep model capacity identical to original frozen N05 TFIDF+LR.
# A user's manual threshold adaptation from external data is forbidden.
TRAIN_ORIG=[(pid,txt(valid[pid]),int(label(valid[pid]))) for pid in train_ids]
assert Counter(y_train)==Counter(v for _,_,v in TRAIN_ORIG)
def prepare_variant(strategy):
    full=list(TRAIN_ORIG)
    if strategy=='orig_only':extra=[]
    elif strategy=='benign_rewrites':extra=[(p,t,y) for p,_,t,y in AUG_ROWS if y==0]
    elif strategy=='all_rewrites':extra=[(p,t,y) for p,_,t,y in AUG_ROWS]
    elif strategy=='gpt4o_only':extra=[(p,t,y) for p,prov,t,y in AUG_ROWS if prov=='gpt4o']
    else:raise ValueError(strategy)
    full.extend(extra)
    parent_counts=Counter(p for p,_,_ in full)
    # Equal parent-weight contribution across original and augmented variants.
    weight=np.array([1/parent_counts[p] for p,_,_ in full],dtype=float)
    return [t for _,t,_ in full],np.asarray([y for _,_,y in full]),weight,len(extra)
STRATEGIES=['orig_only','benign_rewrites','all_rewrites','gpt4o_only']
n11_rows=[];n11_support=[];n11_pairs=[]
n11_original_frozen=model if False else None
for strategy in STRATEGIES:
    text_train,yy,w,extra=prepare_variant(strategy)
    v11=TfidfVectorizer(ngram_range=(1,2),min_df=2,max_df=.98,sublinear_tf=True,
                        max_features=120000,strip_accents='unicode')
    a=v11.fit_transform(text_train)
    clf=LogisticRegression(max_iter=1500,C=1.,class_weight='balanced',random_state=SEED)
    clf.fit(a,yy,sample_weight=w)
    val_probs=clf.predict_proba(v11.transform(X_val))[:,1]
    thr=threshold_at_fpr(y_val,val_probs,.05)
    support={'strategy':strategy,'original_train_parents':len(train_ids),
             'additional_train_rewrites':extra,'training_rows':len(text_train),
             'validation_5pct_fpr_threshold':float(thr)}
    n11_support.append(support)
    comparisons=[('d2_original',X_test,y_test)]
    for provider,records in rewrites.items():
        comparisons.append(('d2_'+provider,[t for _,t,_ in records],
                            np.asarray([yy0 for _,t,yy0 in records],dtype=int)))
    comparisons.append(('ephishllm_english_external',external_x,external_y))
    for variant,texts,truth in comparisons:
        score=clf.predict_proba(v11.transform(texts))[:,1]
        n11_rows.append({'strategy':strategy,'variant':variant,'frozen_d2_val_threshold':float(thr),
                         **score_metrics(truth,score,thr)})
    # D2 held-out source label pairs: false-alarm effect of rewrite.
    org_scores=clf.predict_proba(v11.transform(X_test))[:,1]
    org_by_id={p:float(s) for p,s in zip(test_ids,org_scores)}
    for provider,records in rewrites.items():
        benign=[(p,t) for p,t,y in records if y==0]
        rewrite_scores=clf.predict_proba(v11.transform([t for _,t in benign]))[:,1]
        before=np.array([int(org_by_id[p]>=thr) for p,t in benign])
        after=np.array([int(z>=thr) for z in rewrite_scores])
        n11_pairs.append({'strategy':strategy,'provider':provider,
                          **paired_inference(before,after,seed=SEED,reps=10000)})
# report gain only as descriptive; high recall is the constraint, no selection on external
def n11_csv(filename,rows):
    with (N11_OUT/filename).open('w',encoding='utf-8',newline='') as fd:
        wr=csv.DictWriter(fd,fieldnames=list(rows[0]))
        wr.writeheader();wr.writerows(rows)
n11_csv('support_and_training_counts.csv',n11_support)
n11_csv('augmentation_metrics.csv',n11_rows)
n11_csv('paired_benign_rewrite_effects.csv',n11_pairs)
n11_manifest={'release':'N11-E07-Augmentation-v1.0-2026-11-02',
 'status':'GITHUB_ACTIONS_REAL_DATA_COMPLETED',
 'runner':'GitHub Actions CPU',
 'hypothesis':'Train-parent-only LLM rewrites may reduce out-of-source benign FPR at acceptable phishing recall.',
 'method':'Equal parent weights; original/benign-only/all/GPT-4o-only training variants; TF-IDF LogisticRegression',
 'augmentation_source_drops':dict(AUG_DROPS),
 'train_parent_count':len(train_ids),
 'validation_threshold_source':'D2 original validation only; no E-PhishLLM tuning',
 'caution':['External E-PhishLLM was inspected before designing this experiment; this is exploratory, NOT a fresh blind test',
           'Some LLM-rewritten samples may have changed security intent and are not independently adjudicated',
           'Augmented examples are not extra independent observations; all are grouped by original parent',
           'A new unseen external corpus and transformer comparator are still required for journal-grade claims',
           'Unweighted counts do not establish causal improvement in AI-authored benign email handling'],
 'files':{}}
for path in ['support_and_training_counts.csv','augmentation_metrics.csv','paired_benign_rewrite_effects.csv']:
    n11_manifest['files'][path]=sha256(N11_OUT/path)
(N11_OUT/'run_manifest.json').write_text(json.dumps(n11_manifest,indent=2),encoding='utf-8')
print('N11 completed:',N11_OUT,'augmentation:',len(AUG_ROWS),dict(AUG_DROPS))


## Interpretation rules

Compare external FPR jointly with recall, PR-AUC, and D2 within-source FPR. A lower external false-positive rate accompanied by a large missed-phishing increase is **not** improvement. The trained detector is not an independently validated novel architecture. Subsequent work must include an unseen source, robust near-duplicate control, provenance/rights audit and strong fine-tuned transformer comparator.